In [25]:
'''
task: FOL classification based on MINIFOL2 text
model: t5-base
dataset: FOLIO
evaluation: supervised fine-tuning
'''
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# load FOLIO dataset
import pandas as pd

fol_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_train_sef.csv")
fol_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_test_sef.csv")
fol_valid = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_valid_sef.csv")

In [27]:
# transform data to acceptable format for model training
fol_train = fol_train.applymap(str)
fol_test = fol_test.applymap(str)
fol_valid = fol_valid.applymap(str)

In [28]:
# start preparing for QA pipeline
!pip install datasets
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

In [29]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import T5Tokenizer, T5Model, T5ForConditionalGeneration, T5TokenizerFast

import warnings
warnings.filterwarnings("ignore")

In [30]:
MODEL_NAME = "google/flan-t5-base"
TOKENIZER = T5TokenizerFast.from_pretrained(MODEL_NAME)
MODEL = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, return_dict=True)
OPTIMIZER = Adam(MODEL.parameters(), lr=0.00001)
Q_LEN = 512   # Question Length
T_LEN = 512    # Target Length
BATCH_SIZE = 4
DEVICE = "cuda:0"

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [31]:
class QA_Dataset(Dataset):
    def __init__(self, tokenizer, dataframe, q_len, t_len):
        self.tokenizer = tokenizer
        self.q_len = q_len
        self.t_len = t_len
        self.data = dataframe
        self.subject = self.data["premises-MINIFOL2"]
        self.obj = self.data["conclusion-MINIFOL2"]
        self.relation = self.data['label']

    def __len__(self):
        return len(self.subject)

    def __getitem__(self, idx):
        subject = self.subject[idx]
        obj = self.obj[idx]
        relation = self.relation[idx]

        subject_tokenized = self.tokenizer(subject, obj, max_length=self.q_len, padding="max_length",
                                                    truncation=True, add_special_tokens=True)
        relation_tokenized = self.tokenizer(relation, max_length=self.t_len, padding="max_length",
                                          truncation=True, add_special_tokens=True)

        labels = torch.tensor(relation_tokenized["input_ids"], dtype=torch.long)
        labels[labels == 0] = -100

        return {
            "input_ids": torch.tensor(subject_tokenized["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(subject_tokenized["attention_mask"], dtype=torch.long),
            "labels": labels,
            "decoder_attention_mask": torch.tensor(relation_tokenized["attention_mask"], dtype=torch.long)
        }

In [32]:
train_dataset = QA_Dataset(TOKENIZER, fol_train, Q_LEN, T_LEN)
test_dataset = QA_Dataset(TOKENIZER, fol_test, Q_LEN, T_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [33]:
#torch.cuda.empty_cache()
#del (MODEL)
MODEL.to('cuda')

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [34]:
train_loss = 0
val_loss = 0
train_batch_count = 0
val_batch_count = 0

for epoch in range(5):
    MODEL.train()
    for batch in tqdm(train_loader, desc="Training batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        train_loss += outputs.loss.item()
        train_batch_count += 1

    #Evaluation
    MODEL.eval()
    for batch in tqdm(val_loader, desc="Validation batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        val_loss += outputs.loss.item()
        val_batch_count += 1

    print(f"{epoch+1}/{5} -> Train loss: {train_loss / train_batch_count}\tValidation loss: {val_loss/val_batch_count}")

Validation batches: 100%|██████████| 51/51 [00:13<00:00,  3.78it/s]


1/5 -> Train loss: 2.0941044477373363	Validation loss: 0.41219758870554907


Validation batches: 100%|██████████| 51/51 [00:13<00:00,  3.79it/s]


2/5 -> Train loss: 1.2717885204032064	Validation loss: 0.3729662497838338


Validation batches: 100%|██████████| 51/51 [00:13<00:00,  3.79it/s]


3/5 -> Train loss: 0.9899261981497208	Validation loss: 0.3489352816265393


Validation batches: 100%|██████████| 51/51 [00:13<00:00,  3.78it/s]


4/5 -> Train loss: 0.8428451654687524	Validation loss: 0.33024919405579567


Validation batches: 100%|██████████| 51/51 [00:13<00:00,  3.79it/s]

5/5 -> Train loss: 0.7522564789354801	Validation loss: 0.3128313935416586


In [35]:
# evaluation metrics

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def precision_at_k(actual, predicted):
    act_set = set(actual)
    pred_set = set(predicted)
    result = len(act_set & pred_set) / float(len(predicted))
    return result*100

def apk(actual, predicted, k):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    if not actual:
        return 0.0
    if len(predicted)>k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)
    return score / min(len(actual), k)


def mapk(actual, predicted, k):
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int,
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a, p, k) for a,p in zip(actual, predicted)])


class EvaluationMetrics:

    def __init__(self, ks:list, metric="map") -> None:
        self.ks = ks
        self.metric=metric

    def evaluate(self, actual:list, predicted:list):
        if self.metric == "map":
            return self.MAP(actual, predicted)
        else:
            return self.AP(actual, predicted)

    def MAP(self, actual:list, predicted:list):
        results_dict = {}
        for k in self.ks:
            results_dict["MAP@"+str(k)] = mapk(actual=actual, predicted=predicted, k=k)
        return results_dict

    def AP(self, actual:list, predicted:list):
        results_dict = {}
        for k in self.ks:
            results_dict["AP@"+str(k)] = [apk(actual=actual, predicted=predicted, k=k)
                                          for a, p in zip(actual, predicted)]
        return results_dict

def predict_answer(obj, subject, ref_relation=None):
    inputs = TOKENIZER(subject, obj, return_tensors="pt").to(MODEL.device)

    outputs = MODEL.generate(input_ids=inputs["input_ids"], max_new_tokens=10)

    predicted_relation = TOKENIZER.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    print("premises: \n", subject)
    print("conclusion: \n", obj)
    print("true label: \n", ref_relation)
    print("predicted label: \n", predicted_relation)

    return predicted_relation

In [36]:
# test predictions
syntax = "MINIFOL2"
model = MODEL_NAME.split("/")[-1]
reference_labels = []
predicted_labels = []
for index, row in fol_valid.iterrows():
  conclusion = row[f"conclusion-{syntax}"]
  premises = row[f"premises-{syntax}"]
  label = row["label"]

  predicted_label = predict_answer(conclusion, premises, label)
  reference_labels.append(label)
  predicted_labels.append(predicted_label)

# calculate classification metrics
accuracy_val = accuracy_score(reference_labels, predicted_labels)
precision_val = precision_score(reference_labels, predicted_labels, average="macro")
recall_val = recall_score(reference_labels, predicted_labels, average="macro")
f1_val = f1_score(reference_labels, predicted_labels, average="macro")
classificationreport_val = classification_report(reference_labels, predicted_labels)

# print metrics
print("Accuracy:", accuracy_val)
print("Precision:", precision_val)
print("Recall:", recall_val)
print("F1:", f1_val)
print("Classification Report:", classificationreport_val)

# write results to file
with open(f"folio_{syntax}_{model}_results.txt", "a") as f:
  f.write(f"********** {syntax} {model} INFERENCE COMPLETE **********\n")
  f.write(f"Accuracy: {accuracy_val}\n")
  f.write(f"Precision: {precision_val}\n")
  f.write(f"Recall: {recall_val}\n")
  f.write(f"F1: {f1_val}\n")
  f.write(f"Classification Report: {classificationreport_val}\n")

premises: 
 all:x (inthisclub(x) & performoftenin(x, schooltalentshow) :- attend(x, schoolevent) & veryengagedwith(x, schoolevent))
all:x (inthisclub(x) :- performoftenin(x, schooltalentshow) ^ (inactive(x) & disinterested(x) & memberof(x, community)))
all:x (inthisclub(x) & chaperone(x, highschooldance) :- ~(studen(x) & attendschool(x)))
all:x (inthisclub(x) & (inactive(x) & disinterested(x) & memberof(x, community)) :- chaperone(x, highschooldances))
all:x (inthisclub(x) & (youngchildren(x) ^ teenager(x)) & wishtofurther(x, academiccareer)) :- studen(x) & attendschool(x))
inthisclub(bonnie) & ~((attend(x, schoolevent) & veryengagedwith(bonnie, schoolevent)) ^ (studen(bonne) & attendschool(bonnie)))
conclusion: 
 inthisclub(bonnie) & (perform(bonnie, schooltalentshow))
true label: 
 Uncertain
predicted label: 
 False
premises: 
 all:x (inthisclub(x) & performoftenin(x, schooltalentshow) :- attend(x, schoolevent) & veryengagedwith(x, schoolevent))
all:x (inthisclub(x) :- performoftenin